# NLP Analysis - Financial News Sentiment

This notebook analyzes financial news sentiment and its relationship with stock prices.

## Contents:
1. Load and explore news data
2. Sentiment analysis with VADER
3. Sentiment distribution analysis
4. News-Price relationship
5. Temporal patterns in sentiment

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.append('../src')
from preprocessing.nlp_processor import NLPProcessor

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Libraries imported successfully!")

## 1. Load News Data

In [ ]:
# Load news data
df_news = pd.read_csv('../data/raw/financial_news.csv', parse_dates=['date'])

print(f"Dataset shape: {df_news.shape}")
print(f"\nTickers: {df_news['ticker'].unique()}")
print(f"\nDate range: {df_news['date'].min()} to {df_news['date'].max()}")
print(f"\nSample headlines:")
df_news.head(10)

In [ ]:
# News count by ticker
news_count = df_news.groupby('ticker').size().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
news_count.plot(kind='bar', ax=ax, color='coral')
ax.set_xlabel('Ticker', fontsize=12)
ax.set_ylabel('Number of News Items', fontsize=12)
ax.set_title('News Coverage by Stock', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("\nNews Count by Ticker:")
print(news_count)

## 2. Sentiment Analysis with FinBERT\n\nWe'll use FinBERT, a BERT-based model fine-tuned specifically for financial sentiment analysis.\n\n**FinBERT Advantages:**\n- Trained on 4,000+ financial news articles\n- Understands financial terminology\n- ~92% accuracy on financial sentiment tasks\n- More accurate than general-purpose sentiment models

In [ ]:
# Initialize NLP processor
processor = NLPProcessor(method='vader')

# Process sentiment
df_news = processor.process_news_dataframe(df_news)

print("\nSample with sentiment scores:")
df_news[['headline', 'sentiment_compound', 'sentiment_label']].head(10)

## 3. Sentiment Distribution

In [ ]:
# Overall sentiment distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Compound score distribution
ax1 = axes[0]
ax1.hist(df_news['sentiment_compound'], bins=50, color='skyblue', edgecolor='black', alpha=0.7)
ax1.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Neutral')
ax1.set_xlabel('Compound Sentiment Score', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Distribution of Sentiment Scores', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Sentiment label distribution
ax2 = axes[1]
sentiment_counts = df_news['sentiment_label'].value_counts()
colors = {'positive': 'green', 'neutral': 'gray', 'negative': 'red'}
ax2.bar(sentiment_counts.index, sentiment_counts.values, 
        color=[colors.get(x, 'blue') for x in sentiment_counts.index])
ax2.set_xlabel('Sentiment Label', fontsize=12)
ax2.set_ylabel('Count', fontsize=12)
ax2.set_title('Sentiment Label Distribution', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Sentiment by ticker
fig, ax = plt.subplots(figsize=(12, 6))

df_news.boxplot(column='sentiment_compound', by='ticker', ax=ax)
ax.set_xlabel('Ticker', fontsize=12)
ax.set_ylabel('Compound Sentiment Score', fontsize=12)
ax.set_title('Sentiment Distribution by Stock', fontsize=14, fontweight='bold')
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
plt.suptitle('')
plt.tight_layout()
plt.show()

# Average sentiment by ticker
avg_sentiment = df_news.groupby('ticker')['sentiment_compound'].mean().sort_values()
print("\nAverage Sentiment by Ticker:")
print(avg_sentiment)

## 4. Most Positive and Negative Headlines

In [ ]:
# Most positive headlines
print("Most Positive Headlines:")
print("="*80)
top_positive = df_news.nlargest(10, 'sentiment_compound')[['ticker', 'headline', 'sentiment_compound']]
for idx, row in top_positive.iterrows():
    print(f"\n{row['ticker']}: {row['headline']}")
    print(f"Score: {row['sentiment_compound']:.3f}")

In [ ]:
# Most negative headlines
print("\nMost Negative Headlines:")
print("="*80)
top_negative = df_news.nsmallest(10, 'sentiment_compound')[['ticker', 'headline', 'sentiment_compound']]
for idx, row in top_negative.iterrows():
    print(f"\n{row['ticker']}: {row['headline']}")
    print(f"Score: {row['sentiment_compound']:.3f}")

## 5. Temporal Sentiment Patterns

In [ ]:
# Aggregate daily sentiment
df_daily_sentiment = processor.aggregate_daily_sentiment(df_news)

print("Daily Sentiment Summary:")
df_daily_sentiment.head(10)

In [ ]:
# Plot sentiment over time for each ticker
fig, axes = plt.subplots(len(df_daily_sentiment['ticker'].unique()), 1, 
                         figsize=(14, 4*len(df_daily_sentiment['ticker'].unique())))

if len(df_daily_sentiment['ticker'].unique()) == 1:
    axes = [axes]

for idx, ticker in enumerate(df_daily_sentiment['ticker'].unique()):
    ticker_sentiment = df_daily_sentiment[df_daily_sentiment['ticker'] == ticker].sort_values('date')
    
    ax = axes[idx]
    ax.plot(ticker_sentiment['date'], ticker_sentiment['sentiment_mean'], 
            linewidth=2, color='steelblue', label='Mean Sentiment')
    ax.fill_between(ticker_sentiment['date'], 
                     ticker_sentiment['sentiment_mean'] - ticker_sentiment['sentiment_std'],
                     ticker_sentiment['sentiment_mean'] + ticker_sentiment['sentiment_std'],
                     alpha=0.3, color='steelblue', label='±1 Std Dev')
    ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    ax.set_xlabel('Date', fontsize=11)
    ax.set_ylabel('Sentiment Score', fontsize=11)
    ax.set_title(f'{ticker} - Daily Sentiment Over Time', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Merge with Price Data

In [ ]:
# Load price data
df_prices = pd.read_csv('../data/raw/stock_prices.csv', parse_dates=['Date'])

# Merge sentiment with prices
df_merged = df_prices.merge(
    df_daily_sentiment,
    left_on=['Ticker', 'Date'],
    right_on=['ticker', 'date'],
    how='left'
)

# Calculate price returns
df_merged = df_merged.sort_values(['Ticker', 'Date'])
df_merged['price_return'] = df_merged.groupby('Ticker')['Close'].pct_change() * 100

print(f"Merged dataset shape: {df_merged.shape}")
print(f"\nRows with news: {df_merged['news_count'].notna().sum()}")
df_merged.head()

In [ ]:
# Correlation between sentiment and next day returns
df_merged['next_day_return'] = df_merged.groupby('Ticker')['price_return'].shift(-1)

correlation_data = df_merged[['sentiment_mean', 'price_return', 'next_day_return']].dropna()

print("Correlation Analysis:")
print("\nCorrelation between sentiment and price return:")
print(f"Same day: {correlation_data['sentiment_mean'].corr(correlation_data['price_return']):.4f}")
print(f"Next day: {correlation_data['sentiment_mean'].corr(correlation_data['next_day_return']):.4f}")

# Scatter plot
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].scatter(correlation_data['sentiment_mean'], correlation_data['price_return'], 
                alpha=0.5, s=30)
axes[0].set_xlabel('Sentiment Score', fontsize=12)
axes[0].set_ylabel('Same Day Return (%)', fontsize=12)
axes[0].set_title('Sentiment vs Same Day Return', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.3)
axes[0].axvline(x=0, color='red', linestyle='--', alpha=0.3)

axes[1].scatter(correlation_data['sentiment_mean'], correlation_data['next_day_return'], 
                alpha=0.5, s=30, color='orange')
axes[1].set_xlabel('Sentiment Score', fontsize=12)
axes[1].set_ylabel('Next Day Return (%)', fontsize=12)
axes[1].set_title('Sentiment vs Next Day Return', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.3)
axes[1].axvline(x=0, color='red', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Key Insights

Based on the sentiment analysis:

1. **Sentiment Distribution**: Are most news positive, negative, or neutral?
2. **Ticker Differences**: Do some stocks get more positive/negative coverage?
3. **Sentiment-Price Relationship**: Is there a correlation between sentiment and price movements?
4. **Temporal Patterns**: Are there periods of high negative or positive sentiment?
5. **Predictive Power**: Does sentiment help predict next-day returns?

## Next Steps

- Use these sentiment features in the combined model
- Test different sentiment windows (3-day average, weekly average)
- Experiment with model architectures in notebook 03